# Chunking de Manifestações

Teste de duas configurações do `RecursiveCharacterTextSplitter` nas cinco manifestações mais longas.

In [ ]:

import json, re, unicodedata
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

with open("manifestacoes.json", encoding="utf-8") as f:
    dados = json.load(f)
df = pd.DataFrame(dados)
df["texto"] = df["texto"].fillna("")
textos = df["texto"].tolist()
ids = df["id"].tolist()
print(df.shape)
df.head()


In [ ]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
ordem = df.assign(n_chars=df["texto"].str.len()).sort_values("n_chars", ascending=False).head(5)
ordem[["id","categoria_oficial","n_chars"]]


## Estratégias

In [ ]:

estrategias = [
    {"nome":"A — menor chunk","chunk_size":250,"chunk_overlap":40},
    {"nome":"B — maior contexto","chunk_size":450,"chunk_overlap":80},
]
resultados = {}
for e in estrategias:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=e["chunk_size"], chunk_overlap=e["chunk_overlap"],
        separators=["\n\n","\n",". ","; ",", "," ",""]
    )
    rows=[]
    for _,r in ordem.iterrows():
        chunks=splitter.split_text(r["texto"])
        for k,ch in enumerate(chunks):
            rows.append({"id":r["id"],"chunk":k+1,"texto":ch,"chars":len(ch)})
    resultados[e["nome"]] = pd.DataFrame(rows)
    print("\n",e["nome"], "chunks:",len(rows))
    display(resultados[e["nome"]].head(10))


## Embeddings e PCA dos chunks

In [ ]:

from sentence_transformers import SentenceTransformer
modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
for nome, tabela in resultados.items():
    V = modelo.encode(tabela["texto"].tolist(), normalize_embeddings=True)
    xy = PCA(n_components=2, random_state=42).fit_transform(V)
    tabela["x"], tabela["y"] = xy[:,0], xy[:,1]
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10,7))
    for ident in tabela["id"].unique():
        m=tabela["id"].eq(ident)
        plt.scatter(tabela.loc[m,"x"], tabela.loc[m,"y"], label=ident)
    plt.title(f"PCA dos chunks — {nome}")
    plt.legend()
    plt.show()


## Interpretação

Overlap maior tende a preservar contexto entre o final de um chunk e o início do seguinte, reduzindo cortes abruptos de informação. O custo é gerar mais texto repetido e mais chunks. A escolha final deve equilibrar coesão, redundância e custo computacional; as duas configurações acima permitem observar esse efeito diretamente no corpus.